In [0]:
# Cell: Create all aggregation tables for dashboards
print("="*70)
print("     CREATING GOLD LAYER AGGREGATIONS")
print("="*70)

# Configure for aggregations
spark.conf.set("spark.sql.shuffle.partitions", "50")

# 1. Hourly Metrics
print("\n1/5 Creating agg_hourly_metrics...")
spark.sql("""
    CREATE OR REPLACE TABLE jeit_kg_dev.nyc_tlc.agg_hourly_metrics
    USING DELTA
    AS
    SELECT
        taxi_type,
        pickup_year,
        pickup_month,
        pickup_hour,
        time_of_day,
        COUNT(*) as trip_count,
        ROUND(AVG(trip_distance), 2) as avg_trip_distance,
        ROUND(AVG(trip_duration_minutes), 2) as avg_trip_duration,
        ROUND(AVG(trip_efficiency), 4) as avg_trip_efficiency,
        ROUND(AVG(fare_amount), 2) as avg_fare_amount,
        ROUND(AVG(tip_amount), 2) as avg_tip_amount,
        ROUND(AVG(total_amount), 2) as avg_total_amount,
        ROUND(AVG(fare_per_mile), 2) as avg_fare_per_mile,
        ROUND(SUM(total_amount), 2) as total_revenue,
        ROUND(AVG(passenger_count), 2) as avg_passenger_count
    FROM jeit_kg_dev.nyc_tlc.fact_taxi_trips
    GROUP BY taxi_type, pickup_year, pickup_month, pickup_hour, time_of_day
""")
print("  ✅ agg_hourly_metrics created")

     CREATING GOLD LAYER AGGREGATIONS

1/5 Creating agg_hourly_metrics...
  ✅ agg_hourly_metrics created


In [0]:
# 2. Zone Metrics
print("\n2/5 Creating agg_zone_metrics...")
spark.sql("""
    CREATE OR REPLACE TABLE jeit_kg_dev.nyc_tlc.agg_zone_metrics
    USING DELTA
    AS
    SELECT
        taxi_type,
        pickup_zone_id,
        COUNT(*) as trip_count,
        ROUND(AVG(trip_distance), 2) as avg_trip_distance,
        ROUND(AVG(total_amount), 2) as avg_fare,
        ROUND(SUM(total_amount), 2) as total_revenue
    FROM jeit_kg_dev.nyc_tlc.fact_taxi_trips
    GROUP BY taxi_type, pickup_zone_id
""")
print("  ✅ agg_zone_metrics created")


2/5 Creating agg_zone_metrics...
  ✅ agg_zone_metrics created


In [0]:
# 3. Daily Summary
print("\n3/5 Creating agg_daily_summary...")
spark.sql("""
    CREATE OR REPLACE TABLE jeit_kg_dev.nyc_tlc.agg_daily_summary
    USING DELTA
    AS
    SELECT
        taxi_type,
        CAST(pickup_datetime as DATE) as trip_date,
        pickup_year,
        pickup_month,
        pickup_day_of_week,
        day_of_week_desc,
        COUNT(*) as trip_count,
        ROUND(AVG(trip_distance), 2) as avg_distance,
        ROUND(AVG(trip_duration_minutes), 2) as avg_duration,
        ROUND(AVG(total_amount), 2) as avg_fare,
        ROUND(SUM(total_amount), 2) as total_revenue,
        ROUND(AVG(passenger_count), 2) as avg_passengers
    FROM jeit_kg_dev.nyc_tlc.fact_taxi_trips
    GROUP BY 
        taxi_type,
        CAST(pickup_datetime as DATE),
        pickup_year,
        pickup_month,
        pickup_day_of_week,
        day_of_week_desc
""")
print("  ✅ agg_daily_summary created")


3/5 Creating agg_daily_summary...
  ✅ agg_daily_summary created


In [0]:
# 4. Payment Analysis
print("\n4/5 Creating agg_payment_analysis...")
spark.sql("""
    CREATE OR REPLACE TABLE jeit_kg_dev.nyc_tlc.agg_payment_analysis
    USING DELTA
    AS
    SELECT
        taxi_type,
        payment_type_id,
        payment_type_desc,
        pickup_year,
        COUNT(*) as trip_count,
        ROUND(AVG(total_amount), 2) as avg_fare,
        ROUND(AVG(tip_amount), 2) as avg_tip,
        ROUND(SUM(total_amount), 2) as total_revenue,
        ROUND(AVG(tip_amount) / NULLIF(AVG(fare_amount), 0) * 100, 2) as avg_tip_percentage
    FROM jeit_kg_dev.nyc_tlc.fact_taxi_trips
    WHERE payment_type_id IS NOT NULL
    GROUP BY taxi_type, payment_type_id, payment_type_desc, pickup_year
""")
print("  ✅ agg_payment_analysis created")


4/5 Creating agg_payment_analysis...
  ✅ agg_payment_analysis created


In [0]:
# 5. Rate Code Analysis
print("\n5/5 Creating agg_rate_code_analysis...")
spark.sql("""
    CREATE OR REPLACE TABLE jeit_kg_dev.nyc_tlc.agg_rate_code_analysis
    USING DELTA
    AS
    SELECT
        taxi_type,
        rate_code_id,
        rate_code_desc,
        pickup_year,
        COUNT(*) as trip_count,
        ROUND(AVG(total_amount), 2) as avg_fare,
        ROUND(SUM(total_amount), 2) as total_revenue
    FROM jeit_kg_dev.nyc_tlc.fact_taxi_trips
    WHERE rate_code_id IS NOT NULL
    GROUP BY taxi_type, rate_code_id, rate_code_desc, pickup_year
""")
print("  ✅ agg_rate_code_analysis created")

print("\n" + "="*70)
print("     ALL AGGREGATION TABLES CREATED")
print("="*70)


5/5 Creating agg_rate_code_analysis...
  ✅ agg_rate_code_analysis created

     ALL AGGREGATION TABLES CREATED


In [0]:
# Cell: Create analytical views for common dashboard queries
print("\nCreating analytical views...")

# View 1: Top Pickup Zones with Borough Info
print("\n1/5 Creating vw_top_pickup_zones...")
spark.sql("""
    CREATE OR REPLACE VIEW jeit_kg_dev.nyc_tlc.vw_top_pickup_zones AS
    SELECT 
        z.zone as zone_name,
        z.borough,
        z.service_zone,
        a.taxi_type,
        a.trip_count,
        a.avg_fare,
        a.total_revenue
    FROM jeit_kg_dev.nyc_tlc.agg_zone_metrics a
    JOIN jeit_kg_dev.nyc_tlc.dim_zones z ON a.pickup_zone_id = z.LocationID
""")
print("  ✅ vw_top_pickup_zones created")

# View 2: Peak Hours Analysis
print("\n2/5 Creating vw_peak_hours...")
spark.sql("""
    CREATE OR REPLACE VIEW jeit_kg_dev.nyc_tlc.vw_peak_hours AS
    SELECT 
        h.pickup_hour,
        t.time_of_day,
        t.is_rush_hour,
        h.taxi_type,
        h.pickup_year,
        SUM(h.trip_count) as total_trips,
        ROUND(AVG(h.avg_fare_amount), 2) as avg_fare,
        ROUND(AVG(h.avg_trip_efficiency), 4) as avg_efficiency,
        ROUND(SUM(h.total_revenue), 2) as total_revenue
    FROM jeit_kg_dev.nyc_tlc.agg_hourly_metrics h
    JOIN jeit_kg_dev.nyc_tlc.dim_time t ON h.pickup_hour = t.hour
    GROUP BY h.pickup_hour, t.time_of_day, t.is_rush_hour, h.taxi_type, h.pickup_year
""")
print("  ✅ vw_peak_hours created")

# View 3: Green vs Yellow Comparison
print("\n3/5 Creating vw_green_vs_yellow...")
spark.sql("""
    CREATE OR REPLACE VIEW jeit_kg_dev.nyc_tlc.vw_green_vs_yellow AS
    SELECT 
        taxi_type,
        pickup_year,
        SUM(trip_count) as total_trips,
        ROUND(AVG(avg_distance), 2) as avg_distance,
        ROUND(AVG(avg_duration), 2) as avg_duration,
        ROUND(AVG(avg_fare), 2) as avg_fare,
        ROUND(SUM(total_revenue), 2) as total_revenue
    FROM jeit_kg_dev.nyc_tlc.agg_daily_summary
    GROUP BY taxi_type, pickup_year
""")
print("  ✅ vw_green_vs_yellow created")

# View 4: Day of Week Patterns
print("\n4/5 Creating vw_day_of_week_patterns...")
spark.sql("""
    CREATE OR REPLACE VIEW jeit_kg_dev.nyc_tlc.vw_day_of_week_patterns AS
    SELECT 
        day_of_week_desc,
        pickup_day_of_week,
        taxi_type,
        pickup_year,
        SUM(trip_count) as total_trips,
        ROUND(AVG(avg_fare), 2) as avg_fare,
        ROUND(SUM(total_revenue), 2) as total_revenue
    FROM jeit_kg_dev.nyc_tlc.agg_daily_summary
    GROUP BY day_of_week_desc, pickup_day_of_week, taxi_type, pickup_year
    ORDER BY pickup_day_of_week
""")
print("  ✅ vw_day_of_week_patterns created")

# View 5: Monthly Trends
print("\n5/5 Creating vw_monthly_trends...")
spark.sql("""
    CREATE OR REPLACE VIEW jeit_kg_dev.nyc_tlc.vw_monthly_trends AS
    SELECT 
        pickup_year,
        pickup_month,
        taxi_type,
        SUM(trip_count) as total_trips,
        ROUND(AVG(avg_fare), 2) as avg_fare,
        ROUND(SUM(total_revenue), 2) as total_revenue
    FROM jeit_kg_dev.nyc_tlc.agg_daily_summary
    GROUP BY pickup_year, pickup_month, taxi_type
    ORDER BY pickup_year, pickup_month
""")
print("  ✅ vw_monthly_trends created")

print("\n✅ All analytical views created!")


Creating analytical views...

1/5 Creating vw_top_pickup_zones...
  ✅ vw_top_pickup_zones created

2/5 Creating vw_peak_hours...
  ✅ vw_peak_hours created

3/5 Creating vw_green_vs_yellow...
  ✅ vw_green_vs_yellow created

4/5 Creating vw_day_of_week_patterns...
  ✅ vw_day_of_week_patterns created

5/5 Creating vw_monthly_trends...
  ✅ vw_monthly_trends created

✅ All analytical views created!


In [0]:
# Cell: Gold Layer Complete Summary
print("="*80)
print("                    GOLD LAYER COMPLETE")
print("="*80)

tables = [
    "fact_taxi_trips",
    "dim_zones",
    "dim_time",
    "dim_rate_codes",
    "dim_payment_types",
    "agg_hourly_metrics",
    "agg_zone_metrics",
    "agg_daily_summary",
    "agg_payment_analysis",
    "agg_rate_code_analysis"
]

print("\n📊 Gold Layer Tables:")
for table in tables:
    try:
        count = spark.sql(f"SELECT COUNT(*) FROM jeit_kg_dev.nyc_tlc.{table}").collect()[0][0]
        print(f"  ✅ {table:<30} {count:>15,} records")
    except Exception as e:
        print(f"  ❌ {table:<30} ERROR: {str(e)[:50]}")

print("\n📊 Analytical Views:")
views = [
    "vw_top_pickup_zones",
    "vw_peak_hours",
    "vw_green_vs_yellow",
    "vw_day_of_week_patterns",
    "vw_monthly_trends"
]

for view in views:
    try:
        count = spark.sql(f"SELECT COUNT(*) FROM jeit_kg_dev.nyc_tlc.{view}").collect()[0][0]
        print(f"  ✅ {view:<30} {count:>15,} records")
    except Exception as e:
        print(f"  ❌ {view:<30} ERROR")

print("\n" + "="*80)
print("🎉 GOLD LAYER READY FOR DASHBOARDS!")
print("="*80)

                    GOLD LAYER COMPLETE

📊 Gold Layer Tables:
  ✅ fact_taxi_trips                    104,513,061 records
  ✅ dim_zones                                  265 records
  ✅ dim_time                                    24 records
  ✅ dim_rate_codes                               7 records
  ✅ dim_payment_types                            7 records
  ✅ agg_hourly_metrics                       1,690 records
  ✅ agg_zone_metrics                           517 records
  ✅ agg_daily_summary                        2,131 records
  ✅ agg_payment_analysis                        28 records
  ✅ agg_rate_code_analysis                      40 records

📊 Analytical Views:
  ✅ vw_top_pickup_zones                        517 records
  ✅ vw_peak_hours                              144 records
  ✅ vw_green_vs_yellow                           6 records
  ✅ vw_day_of_week_patterns                     42 records
  ✅ vw_monthly_trends                           71 records

🎉 GOLD LAYER READY FOR DASHBOAR

IOStream.flush timed out
